# Build graph NPZ từ PCAP thô trên Kaggle

**Pipeline:**
```
raw.rar
  → giải nén PCAP/PCAPNG
  → payload_256.npy + metadata.csv           [Stage 1: extract_payload_dataset]
  → teacher_targets.npy                      [Stage 2: build_teacher_targets – SecureBERT]
  → student_cnn_best.pt                      [Stage 3: train_student_cnn]
  → student_embeddings.npy                   [Stage 4: export_student_embeddings]
  → mitre_techniques.csv + mitre_tactics.csv [Stage 5: prepare_mitre_knowledge_base]
  → mitre_techniques_embeddings.npy          [Stage 6: build_mitre_technique_embeddings]
  → graph_artifact_3tier_t082_k5.npz         [Stage 7: build_three_tier_graph_artifact]
```

Mỗi stage bỏ qua tự động nếu output đã tồn tại.  
Upload sẵn các artifact để tiết kiệm thời gian compute.

## Artifact nên upload trước khi chạy

| Dataset Kaggle | File | Bắt buộc |
|---|---|---|
| `nt114-pcap-dataset` | `raw.rar` (hoặc `.pcap`/`.pcapng` trực tiếp) | Có |
| `nt114-student-model` | `student_cnn_best.pt` | Không – bỏ qua stage 2-4 (~3-6h) |
| `nt114-mitre` | `mitre_techniques.csv`, `mitre_tactics.csv`, `mitre_technique_tactic_edges.csv` | Không – bỏ qua stage 5 |
| `nt114-mitre` | `mitre_techniques_embeddings.npy` | Không – bỏ qua stage 6 (~30 phút) |
| `securebert` | folder model SecureBERT | Không – tránh download ~400 MB |


In [ ]:
# ─── CẤU HÌNH – chỉnh ở đây trước khi chạy ─────────────────────────────────
from __future__ import annotations
from pathlib import Path

# Repo GitHub
GITHUB_REPO_URL = "https://github.com/LeThanhPhat-ATTT2023/Do-an-chuyen-nganh_NT114.git"
GITHUB_BRANCH   = "main"
FORCE_RECLONE   = False   # True → xóa và clone lại dù đã có

# Thư mục làm việc (repo clone vào đây, data cũng nằm ở đây)
WORK_DIR   = Path("/kaggle/working/nt114_build_graph")
OUTPUT_ZIP = Path("/kaggle/working/graph_npz_artifact.zip")

# Trích xuất payload từ PCAP
PAYLOAD_LENGTH       = 256
MAX_PACKETS_PER_FILE = None   # int → giới hạn packet/file (test nhanh)
INCLUDE_EMPTY_PAYLOAD = False
STREAM_TO_DISK       = True   # tiết kiệm RAM trên Kaggle
WRITE_BATCH_SIZE     = 100_000

# Teacher (SecureBERT) – chỉ dùng nếu KHÔNG có student_cnn_best.pt
TEACHER_MODEL_NAME  = "ehsanaghaei/SecureBERT"
TEACHER_BATCH_SIZE  = 256
TEACHER_MAX_LENGTH  = 512
TEACHER_NUM_WORKERS = 4
TEACHER_SAVE_DTYPE  = "float16"  # halves disk: ~29.5 GB -> ~14.7 GB
TEACHER_MAX_ROWS    = 5_000_000

# Student CNN – chỉ dùng nếu KHÔNG có student_cnn_best.pt
STUDENT_EPOCHS         = 30
STUDENT_BATCH_SIZE     = 256
STUDENT_NUM_WORKERS    = 2
STUDENT_EMB_BATCH_SIZE = 1024

# MITRE embedding
MITRE_BATCH_SIZE = 64

# Xây dựng đồ thị
FLOW_TIMEOUT_SECONDS  = 30.0
MAX_PACKETS_PER_FLOW  = 20
SIMILARITY_THRESHOLD  = 0.82
PACKET_TOP_K          = 5
FLOW_TOP_K            = 5
SIM_BATCH_SIZE        = 50_000  # T4 16 GB: 50_000 an toàn; OOM → giảm xuống 20_000

print("Config loaded.")
print("WORK_DIR:", WORK_DIR)
print("OUTPUT_ZIP:", OUTPUT_ZIP)

In [ ]:
# ─── KIỂM TRA MÔI TRƯỜNG ────────────────────────────────────────────────────
# Dùng nvidia-smi + subprocess để kiểm tra GPU, không import torch trực tiếp.
# Tránh lỗi "circular import" của PyTorch khi kernel mới khởi động.
import shutil
import subprocess
import sys

print(f"Python: {sys.version}")

# GPU qua nvidia-smi – luôn hoạt động, không phụ thuộc torch
print()
print("=== GPU (nvidia-smi) ===")
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode == 0:
    for i, line in enumerate(smi.stdout.strip().splitlines()):
        print(f"  GPU {i}: {line.strip()}")
else:
    print("  nvidia-smi không phản hồi – GPU chưa được bật.")
    print("  → Settings → Accelerator → GPU T4 x2 → Factory Reset Session.")

# Kiểm tra torch trong subprocess riêng để cô lập circular import
print()
print("=== PyTorch ===")
torch_check = subprocess.run(
    [sys.executable, "-c",
     "import torch; "
     "print('version    :', torch.__version__); "
     "print('cuda       :', torch.cuda.is_available()); "
     "print('device_count:', torch.cuda.device_count())"],
    capture_output=True, text=True,
)
if torch_check.returncode == 0:
    print(torch_check.stdout.strip())
else:
    print("  [WARN] torch import thất bại trong subprocess:")
    print(torch_check.stderr[-600:])
    print()
    print("  FIX: Restart kernel (Run → Restart & Clear Output), sau đó chạy lại từ cell 1.")
    print("  Nếu vẫn lỗi: kiểm tra xem có file tên 'torch.py' trong thư mục làm việc không.")

# Dung lượng disk
print()
disk = shutil.disk_usage("/kaggle/working")
print(f"Disk /kaggle/working – free : {disk.free  / 1024**3:.1f} GB")
print(f"Disk /kaggle/working – total: {disk.total / 1024**3:.1f} GB")

In [ ]:
# ─── KIỂM TRA FILE ĐÃ UPLOAD VÀO /kaggle/input ──────────────────────────────
INPUT_ROOT = Path("/kaggle/input")

CHECK_FILES = [
    "raw.rar",
    "student_cnn_best.pt",
    "mitre_techniques.csv",
    "mitre_tactics.csv",
    "mitre_technique_tactic_edges.csv",
    "mitre_techniques_embeddings.npy",
    "enterprise-attack.json",
]

print("=== File kiểm tra trong /kaggle/input ===")
for name in CHECK_FILES:
    matches = sorted(INPUT_ROOT.rglob(name)) if INPUT_ROOT.exists() else []
    tag     = "[OK]     " if matches else "[MISSING]"
    detail  = str(matches[0]) if matches else "(chưa upload)"
    print(f"  {tag} {name:<48} {detail}")

print()
print("=== Dataset đã upload ===")
if INPUT_ROOT.exists():
    for d in sorted(INPUT_ROOT.iterdir()):
        files = list(d.rglob("*"))
        n   = sum(1 for f in files if f.is_file())
        sz  = sum(f.stat().st_size for f in files if f.is_file())
        print(f"  {d.name:<45} {n:>4} files  {sz / 1024**3:.3f} GB")

In [ ]:
# ─── CLONE REPO + CÀI PACKAGE ───────────────────────────────────────────────
import os
import subprocess

def run(cmd: list, cwd: Path | None = None) -> None:
    """Chạy lệnh shell, in lệnh trước khi chạy."""
    print("\n$", " ".join(str(x) for x in cmd), flush=True)
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True)


def is_repo(path: Path) -> bool:
    return (
        (path / "src" / "graphslm_ids").exists()
        and (path / "configs" / "hgt_t082_k5_l3_d01.yaml").exists()
    )


# 1. Clone hoặc cập nhật repo
if FORCE_RECLONE and WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
    print("Đã xóa WORK_DIR cũ để clone lại.")

if is_repo(WORK_DIR):
    print(f"Repo đã tồn tại tại {WORK_DIR}")
    run(["git", "-C", str(WORK_DIR), "pull"])
else:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    clone_cmd = ["git", "clone"]
    if GITHUB_BRANCH:
        clone_cmd += ["--branch", GITHUB_BRANCH]
    clone_cmd += [GITHUB_REPO_URL, str(WORK_DIR)]
    run(clone_cmd)

assert is_repo(WORK_DIR), f"Clone thất bại: không tìm thấy src/graphslm_ids trong {WORK_DIR}"

# 2. Cài các package chưa có trên Kaggle
# torch đã có sẵn; bỏ qua để không bị downgrade
run([
    sys.executable, "-m", "pip", "install", "-q",
    "scapy>=2.5.0",
    "dpkt>=1.9.8",
    "transformers>=4.42",
    "sentencepiece",
    "accelerate",
    "pyyaml>=6.0",
])

# 3. Cài repo dưới dạng editable package
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(WORK_DIR)])

# Chuyển cwd sang WORK_DIR để các module dùng đường dẫn tương đối
os.chdir(WORK_DIR)
print("\nCWD:", Path.cwd())
print("Clone + install hoàn tất.")

In [ ]:
# ─── TỰ ĐỘNG PHÁT HIỆN & COPY ARTIFACT TỪ /kaggle/input ────────────────────
# Cell này quét /kaggle/input, copy mọi file đã upload vào đúng vị trí trong
# WORK_DIR. Các stage sau đó tự động bỏ qua nếu output đã tồn tại.
# ACTIVE_PCAP_GLOBS chỉ cần thiết nếu payload_256.npy chưa có.

import glob as _glob
import shutil as _shutil
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

# Đường dẫn đích trong WORK_DIR
_PAYLOAD_DIR = WORK_DIR / "data" / "interim" / "payload_dataset"
_PROCESSED   = WORK_DIR / "data" / "processed"
_MITRE_DIR   = WORK_DIR / "data" / "mitre"
_STUDENT_DIR = WORK_DIR / "outputs" / "student_cnn"

for _d in [_PAYLOAD_DIR, _PROCESSED, _MITRE_DIR, _STUDENT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)


def find_in_input(name: str) -> "Path | None":
    """Tìm file đầu tiên khớp tên trong /kaggle/input (tìm đệ quy)."""
    if not INPUT_ROOT.exists():
        return None
    matches = sorted(INPUT_ROOT.rglob(name))
    return matches[0] if matches else None


def copy_if_missing(name: str, dest: Path) -> bool:
    """Copy file từ /kaggle/input vào dest nếu dest chưa tồn tại.
    Trả về True nếu dest tồn tại sau thao tác."""
    if dest.exists():
        return True
    src = find_in_input(name)
    if src:
        _shutil.copy2(src, dest)
        print(f"  [COPY] {name:<48} → {dest.relative_to(WORK_DIR)}")
        return True
    return False


print("=" * 60)
print("Bước 0: Tự động copy artifact từ /kaggle/input")
print("=" * 60)

# ── Payload dataset (Stage 1 output) ──────────────────────────────────────────
print()
print("[ Payload dataset ]")
copy_if_missing("payload_256.npy", _PAYLOAD_DIR / "payload_256.npy")
copy_if_missing("metadata.csv",    _PAYLOAD_DIR / "metadata.csv")

# ── Student model (bỏ qua Stage 2-4 nếu có) ───────────────────────────────────
print()
print("[ Student model ]")
copy_if_missing("student_cnn_best.pt",    _STUDENT_DIR / "student_cnn_best.pt")
copy_if_missing("teacher_targets.npy",    _PROCESSED   / "teacher_targets.npy")
copy_if_missing("student_embeddings.npy", _PROCESSED   / "student_embeddings.npy")

# ── MITRE knowledge (Stage 5-6) ───────────────────────────────────────────────
print()
print("[ MITRE knowledge ]")
copy_if_missing("mitre_techniques.csv",             _MITRE_DIR / "mitre_techniques.csv")
copy_if_missing("mitre_tactics.csv",                _MITRE_DIR / "mitre_tactics.csv")
copy_if_missing("mitre_technique_tactic_edges.csv", _MITRE_DIR / "mitre_technique_tactic_edges.csv")
copy_if_missing("enterprise-attack.json",           _MITRE_DIR / "enterprise-attack.json")
copy_if_missing("mitre_techniques_embeddings.npy",  _MITRE_DIR / "mitre_techniques_embeddings.npy")

# ── Graph NPZ (Stage 7 output) ─────────────────────────────────────────────────
print()
print("[ Graph NPZ (nếu đã có) ]")
copy_if_missing("graph_artifact_3tier_t082_k5.npz",
                _PROCESSED / "graph_artifact_3tier_t082_k5.npz")
copy_if_missing("graph_artifact_3tier_t082_k5.meta.json",
                _PROCESSED / "graph_artifact_3tier_t082_k5.meta.json")

# ── SecureBERT local model ─────────────────────────────────────────────────────
print()
print("[ SecureBERT model ]")
SECUREBERT_LOCAL: "str | None" = None
if INPUT_ROOT.exists():
    for _cfg in sorted(INPUT_ROOT.rglob("config.json")):
        _cfg_text = _cfg.read_text(encoding="utf-8", errors="ignore")
        if "securebert" in str(_cfg).lower() or "SecureBERT" in _cfg_text:
            SECUREBERT_LOCAL = str(_cfg.parent)
            print(f"  [OK] SecureBERT local: {SECUREBERT_LOCAL}")
            break
    if SECUREBERT_LOCAL is None:
        print("  [MISSING] SecureBERT local – sẽ download từ HuggingFace khi cần.")

# ── PCAP / RAR (chỉ xử lý nếu payload chưa có) ───────────────────────────────
print()
PCAP_EXTRACT_DIR  = WORK_DIR / "data" / "raw_pcap"
ACTIVE_PCAP_GLOBS: "list[str]" = []

_payload_ready = (
    (_PAYLOAD_DIR / "payload_256.npy").exists()
    and (_PAYLOAD_DIR / "metadata.csv").exists()
)

if _payload_ready:
    print("[ PCAP/RAR ] Payload đã có – bỏ qua tìm PCAP/RAR.")
else:
    print("[ PCAP/RAR ] Payload chưa có – tìm PCAP/RAR ...")
    _pcap_search = [
        str(PCAP_EXTRACT_DIR / "**" / "*.pcap"),
        str(PCAP_EXTRACT_DIR / "**" / "*.pcapng"),
        "/kaggle/input/**/*.pcap",
        "/kaggle/input/**/*.pcapng",
    ]
    _found: dict[str, Path] = {}
    for _pat in _pcap_search:
        for _p in _glob.glob(_pat, recursive=True):
            _pp = Path(_p)
            if _pp.suffix.lower() in {".pcap", ".pcapng"} and _pp.is_file():
                _found[str(_pp.resolve())] = _pp
    _pcaps = sorted(_found.values())

    if _pcaps:
        print(f"  [OK] Tìm thấy {len(_pcaps)} PCAP file.")
        ACTIVE_PCAP_GLOBS = [
            str(PCAP_EXTRACT_DIR / "**" / "*.pcap"),
            str(PCAP_EXTRACT_DIR / "**" / "*.pcapng"),
            "/kaggle/input/**/*.pcap",
            "/kaggle/input/**/*.pcapng",
        ]
    else:
        _rar_list = sorted(INPUT_ROOT.rglob("raw.rar")) if INPUT_ROOT.exists() else []
        if _rar_list:
            _rar = _rar_list[0]
            PCAP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
            print(f"  Giải nén {_rar.name} → {PCAP_EXTRACT_DIR} ...")

            def run(cmd: list, cwd: "Path | None" = None) -> None:  # noqa: F811
                print("  $", " ".join(str(x) for x in cmd), flush=True)
                import subprocess as _sp
                _sp.run([str(x) for x in cmd],
                        cwd=str(cwd) if cwd else None, check=True)

            _has7z = subprocess.run(["which", "7z"], capture_output=True).returncode == 0
            if _has7z:
                run(["7z", "x", str(_rar), f"-o{PCAP_EXTRACT_DIR}", "-y"])
            else:
                subprocess.run(["apt-get", "install", "-y", "-q", "unrar-free"], check=True)
                run(["unrar-free", "x", str(_rar), str(PCAP_EXTRACT_DIR) + "/"])
            ACTIVE_PCAP_GLOBS = [
                str(PCAP_EXTRACT_DIR / "**" / "*.pcap"),
                str(PCAP_EXTRACT_DIR / "**" / "*.pcapng"),
            ]
        else:
            print("  [WARN] Không có PCAP/RAR – Stage 1 sẽ báo lỗi nếu payload vẫn thiếu.")

# ── Tổng kết ──────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("Tổng kết – file sẵn sàng trong WORK_DIR:")
print("=" * 60)
_summary = [
    ("payload_256.npy",                 _PAYLOAD_DIR / "payload_256.npy"),
    ("metadata.csv",                    _PAYLOAD_DIR / "metadata.csv"),
    ("student_cnn_best.pt",             _STUDENT_DIR / "student_cnn_best.pt"),
    ("teacher_targets.npy",             _PROCESSED   / "teacher_targets.npy"),
    ("student_embeddings.npy",          _PROCESSED   / "student_embeddings.npy"),
    ("mitre_techniques.csv",            _MITRE_DIR   / "mitre_techniques.csv"),
    ("mitre_techniques_embeddings.npy", _MITRE_DIR   / "mitre_techniques_embeddings.npy"),
    ("graph_artifact_3tier_t082_k5.npz",_PROCESSED   / "graph_artifact_3tier_t082_k5.npz"),
]
for _name, _path in _summary:
    _tag = "[OK]    " if _path.exists() else "[MISSING]"
    print(f"  {_tag} {_name}")

In [ ]:
# ─── STAGE 1: Extract payload dataset từ PCAP ────────────────────────────────
# Output: data/interim/payload_dataset/payload_256.npy + metadata.csv
# Bỏ qua tự động nếu file đã có (copy từ /kaggle/input hoặc chạy trước đó).

PAYLOAD_DIR  = WORK_DIR / "data" / "interim" / "payload_dataset"
PAYLOAD_NPY  = PAYLOAD_DIR / "payload_256.npy"
METADATA_CSV = PAYLOAD_DIR / "metadata.csv"

if PAYLOAD_NPY.exists() and METADATA_CSV.exists():
    _npy_mb  = PAYLOAD_NPY.stat().st_size  / 1024**2
    _csv_mb  = METADATA_CSV.stat().st_size / 1024**2
    _n_rows  = sum(1 for _ in open(METADATA_CSV)) - 1
    print(f"[SKIP] payload_256.npy  ({_npy_mb:.1f} MB)")
    print(f"[SKIP] metadata.csv     ({_n_rows:,} packet, {_csv_mb:.1f} MB)")
else:
    # ACTIVE_PCAP_GLOBS được set bởi cell auto-detect ở trên.
    # Fallback nếu cell đó chưa chạy:
    if "ACTIVE_PCAP_GLOBS" not in dir() or not ACTIVE_PCAP_GLOBS:
        raise RuntimeError(
            "ACTIVE_PCAP_GLOBS chưa được set.\n"
            "Hãy chạy cell 'Tự động phát hiện & copy artifact' trước,\n"
            "hoặc upload payload-dataset (payload_256.npy + metadata.csv) vào /kaggle/input."
        )

    PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)
    print("Stage 1: Extract payload từ PCAP ...")
    print(f"  PCAP globs: {ACTIVE_PCAP_GLOBS}")
    cmd = [
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.preprocessing.extract_payload_dataset",
        "--input-glob", *ACTIVE_PCAP_GLOBS,
        "--output-dir",       str(PAYLOAD_DIR),
        "--payload-length",   str(PAYLOAD_LENGTH),
        "--write-batch-size", str(WRITE_BATCH_SIZE),
    ]
    if STREAM_TO_DISK:
        cmd.append("--stream-to-disk")
    if INCLUDE_EMPTY_PAYLOAD:
        cmd.append("--include-empty-payload")
    if MAX_PACKETS_PER_FILE is not None:
        cmd += ["--max-packets-per-file", str(MAX_PACKETS_PER_FILE)]
    run(cmd, cwd=WORK_DIR)

print()
print("payload_256.npy:", PAYLOAD_NPY,  f"({PAYLOAD_NPY.stat().st_size  / 1024**3:.3f} GB)")
print("metadata.csv   :", METADATA_CSV, f"({METADATA_CSV.stat().st_size / 1024**2:.1f} MB)")

In [ ]:
# ─── STAGE 2: Build teacher targets (SecureBERT) ─────────────────────────────
# Output: data/processed/teacher_targets.npy
# Bỏ qua nếu student_cnn_best.pt đã có (không cần teacher để tiếp tục).

PROCESSED_DIR = WORK_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_NPY  = PROCESSED_DIR / "teacher_targets.npy"
STUDENT_DIR  = WORK_DIR / "outputs" / "student_cnn"
STUDENT_CKPT = STUDENT_DIR / "student_cnn_best.pt"

if STUDENT_CKPT.exists():
    print(f"[SKIP] student_cnn_best.pt đã có → không cần teacher targets.")
elif TEACHER_NPY.exists():
    print(f"[SKIP] teacher_targets.npy đã có ({TEACHER_NPY.stat().st_size / 1024**3:.3f} GB)")
else:
    # Dùng SecureBERT local nếu đã detect được, ngược lại download
    _model = getattr(__builtins__, "__dict__", {}).get("SECUREBERT_LOCAL") or globals().get("SECUREBERT_LOCAL") or TEACHER_MODEL_NAME
    print(f"Stage 2: Build teacher targets – model: {_model} ...")
    print("  Thời gian dự kiến: 2-4 giờ (upload student_cnn_best.pt để bỏ qua).")
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.preprocessing.build_teacher_targets",
        "--payload-npy",  str(PAYLOAD_NPY),
        "--metadata-csv", str(METADATA_CSV),
        "--output-path",  str(TEACHER_NPY),
        "--model-name",   _model,
        "--batch-size",   str(TEACHER_BATCH_SIZE),
        "--max-length",   str(TEACHER_MAX_LENGTH),
        "--num-workers",  str(TEACHER_NUM_WORKERS),
        "--save-dtype",   TEACHER_SAVE_DTYPE,
        "--max-rows",     str(TEACHER_MAX_ROWS),
        "--device",       "auto",
    ], cwd=WORK_DIR)

if TEACHER_NPY.exists():
    print("teacher_targets.npy:", TEACHER_NPY,
          f"({TEACHER_NPY.stat().st_size / 1024**3:.3f} GB)")

In [ ]:
# ─── STAGE 3: Train student CNN ──────────────────────────────────────────────
# Output: outputs/student_cnn/student_cnn_best.pt
# Bỏ qua nếu checkpoint đã có

if STUDENT_CKPT.exists():
    print(f"[SKIP] student_cnn_best.pt đã có ({STUDENT_CKPT.stat().st_size / 1024**2:.1f} MB)")
else:
    if not TEACHER_NPY.exists():
        raise FileNotFoundError(
            "Cần teacher_targets.npy để train student CNN.\n"
            "Stage 2 phải hoàn thành trước, hoặc upload student_cnn_best.pt sẵn."
        )
    STUDENT_DIR.mkdir(parents=True, exist_ok=True)
    print("Stage 3: Train student CNN ...")
    run([
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.training.train_student_cnn",
        "--payload-npy",   str(PAYLOAD_NPY),
        "--teacher-npy",   str(TEACHER_NPY),
        "--output-dir",    str(STUDENT_DIR),
        "--epochs",        str(STUDENT_EPOCHS),
        "--batch-size",    str(STUDENT_BATCH_SIZE),
        "--num-workers",   str(STUDENT_NUM_WORKERS),
        "--device",        "auto",
    ], cwd=WORK_DIR)

print("student_cnn_best.pt:", STUDENT_CKPT)

In [ ]:
# ─── STAGE 4: Export student embeddings ─────────────────────────────────────
# Output: data/processed/student_embeddings.npy

STUDENT_EMB_NPY = PROCESSED_DIR / "student_embeddings.npy"

if STUDENT_EMB_NPY.exists():
    print(f"[SKIP] student_embeddings.npy đã có ({STUDENT_EMB_NPY.stat().st_size / 1024**3:.3f} GB)")
else:
    print("Stage 4: Export student embeddings ...")
    run([
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.training.export_student_embeddings",
        "--payload-npy",   str(PAYLOAD_NPY),
        "--checkpoint",    str(STUDENT_CKPT),
        "--output-path",   str(STUDENT_EMB_NPY),
        "--batch-size",    str(STUDENT_EMB_BATCH_SIZE),
        "--device",        "auto",
    ], cwd=WORK_DIR)

print("student_embeddings.npy:", STUDENT_EMB_NPY, f"({STUDENT_EMB_NPY.stat().st_size / 1024**3:.3f} GB)")

In [ ]:
# ─── STAGE 5: Prepare MITRE knowledge base ───────────────────────────────────
# Output: data/mitre/mitre_techniques.csv
#         data/mitre/mitre_tactics.csv
#         data/mitre/mitre_technique_tactic_edges.csv
# Bỏ qua nếu đã có (hoặc copy từ /kaggle/input)

MITRE_DIR  = WORK_DIR / "data" / "mitre"
MITRE_DIR.mkdir(parents=True, exist_ok=True)

MITRE_TECH  = MITRE_DIR / "mitre_techniques.csv"
MITRE_TAC   = MITRE_DIR / "mitre_tactics.csv"
MITRE_EDGE  = MITRE_DIR / "mitre_technique_tactic_edges.csv"
MITRE_STIX  = MITRE_DIR / "enterprise-attack.json"


def copy_mitre_csv(name: str, dest: Path) -> bool:
    """Copy file MITRE từ /kaggle/input nếu chưa có ở WORK_DIR."""
    if dest.exists():
        return True
    src = find_in_input(name)
    if src:
        import shutil as _sh
        _sh.copy2(src, dest)
        print(f"  Copy {name}: {src} → {dest}")
        return True
    return False


# Thử lấy từ /kaggle/input trước
has_tech  = copy_mitre_csv("mitre_techniques.csv", MITRE_TECH)
has_tac   = copy_mitre_csv("mitre_tactics.csv", MITRE_TAC)
has_edge  = copy_mitre_csv("mitre_technique_tactic_edges.csv", MITRE_EDGE)

if has_tech and has_tac and has_edge:
    print("[SKIP] MITRE CSV đã có đầy đủ – bỏ qua stage 5.")
else:
    # Cần enterprise-attack.json để build từ đầu
    if not MITRE_STIX.exists():
        stix_src = find_in_input("enterprise-attack.json")
        if stix_src:
            import shutil as _sh
            _sh.copy2(stix_src, MITRE_STIX)
            print(f"  Copy enterprise-attack.json: {stix_src} → {MITRE_STIX}")
        else:
            raise FileNotFoundError(
                "Thiếu enterprise-attack.json.\n"
                "Tải từ https://github.com/mitre/cti (enterprise-attack.json) "
                "và upload vào /kaggle/input, hoặc upload mitre_*.csv trực tiếp."
            )
    print("Stage 5: Build MITRE knowledge base từ enterprise-attack.json ...")
    run([
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.preprocessing.prepare_mitre_knowledge_base",
        "--input-json",                  str(MITRE_STIX),
        "--techniques-csv",              str(MITRE_TECH),
        "--tactics-csv",                 str(MITRE_TAC),
        "--technique-tactic-edges-csv",  str(MITRE_EDGE),
        "--stats-json",                  str(MITRE_DIR / "mitre_export_stats.json"),
    ], cwd=WORK_DIR)

print()
for f in [MITRE_TECH, MITRE_TAC, MITRE_EDGE]:
    rows = sum(1 for _ in open(f)) - 1
    print(f"  {f.name:<45} {rows:>6} rows")

In [ ]:
# ─── STAGE 6: Build MITRE technique embeddings ───────────────────────────────
# Output: data/mitre/mitre_techniques_embeddings.npy
# Bỏ qua nếu đã có (copy từ /kaggle/input hoặc chạy trước đó).

MITRE_DIR = WORK_DIR / "data" / "mitre"
MITRE_TECH = MITRE_DIR / "mitre_techniques.csv"
MITRE_EMB  = MITRE_DIR / "mitre_techniques_embeddings.npy"

if MITRE_EMB.exists():
    import numpy as _np
    _shape = _np.load(MITRE_EMB).shape
    print(f"[SKIP] mitre_techniques_embeddings.npy đã có – shape={_shape}")
else:
    # Ưu tiên SecureBERT local (detect từ cell auto-detect)
    _model = globals().get("SECUREBERT_LOCAL") or TEACHER_MODEL_NAME
    print(f"Stage 6: Build MITRE technique embeddings – model: {_model} ...")

    cmd = [
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.preprocessing.build_mitre_technique_embeddings",
        "--techniques-csv", str(MITRE_TECH),
        "--output-path",    str(MITRE_EMB),
        "--model-name",     _model,
        "--batch-size",     str(MITRE_BATCH_SIZE),
        "--device",         "auto",
    ]
    run(cmd, cwd=WORK_DIR)

import numpy as _np2
_emb = _np2.load(MITRE_EMB)
print(f"mitre_techniques_embeddings.npy: shape={_emb.shape}, dtype={_emb.dtype}")

In [ ]:
# ─── STAGE 7: Build three-tier graph NPZ ─────────────────────────────────────
# Output: data/processed/graph_artifact_3tier_t082_k5.npz
#         data/processed/graph_artifact_3tier_t082_k5.meta.json

GRAPH_NPZ       = PROCESSED_DIR / "graph_artifact_3tier_t082_k5.npz"
GRAPH_META_JSON = PROCESSED_DIR / "graph_artifact_3tier_t082_k5.meta.json"

if GRAPH_NPZ.exists() and GRAPH_META_JSON.exists():
    print(f"[SKIP] graph_artifact_3tier_t082_k5.npz đã có ({GRAPH_NPZ.stat().st_size / 1024**3:.3f} GB)")
else:
    # Kiểm tra đủ input
    for required_path in [METADATA_CSV, PAYLOAD_NPY, STUDENT_EMB_NPY, MITRE_TECH, MITRE_EMB, MITRE_EDGE]:
        if not required_path.exists():
            raise FileNotFoundError(f"Thiếu input: {required_path}")

    print(f"Stage 7: Build three-tier graph NPZ (threshold={SIMILARITY_THRESHOLD}, top_k={PACKET_TOP_K}) ...")
    print(f"  SIM_BATCH_SIZE={SIM_BATCH_SIZE} – nếu OOM, giảm xuống 20_000 trong cell Config.")
    run([
        sys.executable, "-u", "-m",
        "graphslm_ids.offline_path.preprocessing.build_three_tier_graph_artifact",
        "--metadata-csv",                   str(METADATA_CSV),
        "--payload-npy",                    str(PAYLOAD_NPY),
        "--student-embedding-npy",          str(STUDENT_EMB_NPY),
        "--mitre-techniques-csv",           str(MITRE_TECH),
        "--mitre-technique-embeddings-npy", str(MITRE_EMB),
        "--mitre-technique-tactic-edges-csv", str(MITRE_EDGE),
        "--output-npz",                     str(GRAPH_NPZ),
        "--output-meta-json",               str(GRAPH_META_JSON),
        "--flow-timeout-seconds",           str(FLOW_TIMEOUT_SECONDS),
        "--max-packets-per-flow",           str(MAX_PACKETS_PER_FLOW),
        "--similarity-threshold",           str(SIMILARITY_THRESHOLD),
        "--packet-top-k",                   str(PACKET_TOP_K),
        "--flow-top-k",                     str(FLOW_TOP_K),
        "--sim-batch-size",                 str(SIM_BATCH_SIZE),
        "--device",                         "auto",
    ], cwd=WORK_DIR)

print()
print("graph_artifact_3tier_t082_k5.npz      :", GRAPH_NPZ, f"({GRAPH_NPZ.stat().st_size / 1024**3:.3f} GB)")
print("graph_artifact_3tier_t082_k5.meta.json:", GRAPH_META_JSON)

In [ ]:
# ─── KIỂM TRA OUTPUT + ĐÓNG GÓI ZIP ─────────────────────────────────────────
import json
import zipfile

# In thống kê từ meta.json
if GRAPH_META_JSON.exists():
    meta = json.loads(GRAPH_META_JSON.read_text(encoding="utf-8"))
    print("=== Graph stats ===")
    stat_keys = [
        "num_packets", "num_flows", "num_techniques", "num_tactics",
        "num_packet_technique_edges", "num_flow_technique_edges",
        "similarity_threshold", "packet_top_k", "flow_top_k",
        "device", "sim_batch_size",
    ]
    for k in stat_keys:
        if k in meta:
            print(f"  {k:<35} {meta[k]}")

print()
print("=== File output ===")
for f in [GRAPH_NPZ, GRAPH_META_JSON, STUDENT_EMB_NPY, MITRE_EMB]:
    if f.exists():
        print(f"  {f.name:<48} {f.stat().st_size / 1024**2:>10.1f} MB")

# Đóng gói artifact vào zip
print()
print("Đóng gói artifact ...")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

artifact_files = [
    GRAPH_NPZ,
    GRAPH_META_JSON,
    MITRE_TECH,
    MITRE_TAC,
    MITRE_EDGE,
    MITRE_EMB,
]

with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    for f in artifact_files:
        if f.exists():
            try:
                arc = f.relative_to(WORK_DIR)
            except ValueError:
                arc = Path(f.name)
            zf.write(f, arc)
            print(f"  + {arc}")

print()
print("OUTPUT_ZIP:", OUTPUT_ZIP)
print("Size      :", round(OUTPUT_ZIP.stat().st_size / 1024**3, 3), "GB")
print()
print("Hoàn tất! Tải file", OUTPUT_ZIP.name, "về để dùng trong notebook train HGT.")